# MPPP — 03 COLMAP alignment (Navcam, optionally + Mastcam-Z) (MPPP 0.63.0)

PDS products in, a prior-aligned COLMAP block out: images processed and masked, features matched, a weighted bundle adjustment from the rover poses (stations with fewer than `LOCALIZE_MIN_IMAGES` images get no waypoint position prior), a health report and the `error_input/` folder for notebooks 04 (camera models) and 05 (error analysis). Every setting is in the `parameters` cell (section 0); `scripts/run_sites.py` runs this notebook for several sites by overriding them.

**Sites** (`SITES`: name → sol range, from `mppp/data/sites.json` since v0p43; add your own there or in `SITES_EXTRA`):

| site | sols | site | sols |
|---|---|---|---|
| butler_landing | 1–14 | airey_hill | 960–991 |
| van_zyl | 49–71 | bunsen_peak | 1066–1095 |
| rochette | 178–190 | overlook_mountain | 1150–1155 |
| seitah_north | 238–279 | pearce_canyon | 1183–1218 |
| sid | 361–378 | pico_turquino | 1307–1310 |
| rose_river_falls | 448–450 | rio_chiquito | 1333–1337 |
| rockytop (difficult: winter sun) | 461–530 | south_arm | 1408–1412 |
| enchanted_lake | 556–590 | bell_island (cross-station packing) | 1451–1467 |
| whale_mountain | 606–610 | taylorfjellet_large (widest LMST range) | 1601–1645 |
| threeforks_south | 652–674 | taylorfjellet | 1606–1625 |
| threeforks (easy: flat ground) | 652–692 | origny_large | 1765–1813 |
| threeforks_north | 680–692 | origny | 1781–1813 |
| knob_mountain | 698–706 | olifants | 1880–1889 |
| berea | 732–738 | groloy | 1922–1934 |
| belva_crater | 784–815 | hippo_pools | 1947–1955 |
| tuxedo_park | 897–908 | marble_mountain | 1965–1979 |

`ADD_NEARBY_WAYPOINTS` adds every other visit within that many metres of a station of the sol range (0 = off).

**What the run does:** select one product per exposure → process (8-bit PNG, terrain mask, CAHV + waypoint pose; images pointing above `SKY_ELEVATION_DEG`, taken outside `LMST_WINDOW_H`, or with more than `MAX_SATURATED_FRACTION` saturated pixels are left out) → one COLMAP camera per eye (rational Navcam lens, consensus start cameras and rig; the stereo baseline is held and sets the scale) → SIFT at native resolution inside the masks → exhaustive matching → triangulation from the rover poses and a weighted bundle adjustment with position and attitude priors (frames that do not align with the majority are excluded; with Mastcam-Z, the Navcam block is solved first and held) → health report with a top-down station map → export.

**Frame.** The block lives in East–North–Up metres of the landing site frame minus a rounded offset (`proj.offset`, recorded in `project.json` and `error_input/summary.json`). The attitude priors (`ATTITUDE_PRIOR_DEG`) keep the cardinal directions; the health check `block_rotation_deg` reports any common rotation of the block away from them.

**Reruns (v0p43).** `SOURCE = "processed"` aligns the images already in `WORK/processed` without the PDS archive. Leave images out with `WORK/exclude_images.txt` (stations, sols, sequences or file-name patterns). `VARIANT = "name"` writes to `WORK/colmap_name`, so other SfM settings or camera models sit beside the default `colmap/` results. The features are reused while the images are the same, and the matches while the features and matching settings are the same. Without Jupyter: `align_here.bat` in a WORK folder, or `scripts/align_scape.py` and `scripts/run_sites.py`.

**COLMAP GUI.** `open_in_colmap.bat` in the project folder starts COLMAP (`COLMAP_BAT`) with the database, the images and the refined model loaded.

This kernel needs pip `pycolmap` 4.2 and `pyceres` 2.6 (`pip install -e .[sfm]`); GPU extraction and matching run in `GPU_PY`.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows/conda: duplicate Intel OpenMP guard; before torch
import sys, json
from pathlib import Path
# use the MPPP source this notebook belongs to (the folder above it that holds src/mppp), else an installed mppp
_here = Path.cwd().resolve()
_root = next((p for p in (_here, *_here.parents) if (p / "src" / "mppp").is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root / "src"))
import mppp
import pycolmap, pyceres
from mppp.select import find_imgs
from mppp.sfm import (SfmProject, select_best_products, extract_features, build_database, prior_overlap_pairs,
                      match, reconstruct, export_for_error, check_gpu_python, check_ba_environment)
from mppp.sfm.health import assess_alignment, health_table, write_health
from mppp.sfm.reconstruction import convergence_statistics
print("MPPP", mppp.__version__, "from", Path(mppp.__file__).parent, "| pycolmap", pycolmap.__version__,
      "(CUDA:", pycolmap.has_cuda, ") | pyceres", pyceres.__version__)
check_ba_environment()        # stops here if this kernel cannot run the weighted BA (e.g. it is the conda CUDA env)

## 0. Settings
Every setting of the run is in the cell below. `scripts/run_sites.py` and `scripts/align_scape.py` override them, one site at a time.

In [ ]:
# --- data and output ------------------------------------------------------------------------------------
PDS_DIR     = Path("D:/data/m2020")          # the local PDS archive (searched recursively)
SCAPES_ROOT = Path("D:/scapes/colmap")       # one WORK folder per site below this
# Optional GPU for feature extraction and matching (Windows): the python.exe of a second environment with the
# conda-forge CUDA pycolmap of the same COLMAP version (README, "GPU on Windows").  None = this kernel, CPU.
GPU_PY = r"C:\Users\ctate\AppData\Local\miniconda3\envs\mppp_gpu\python.exe"
COLMAP_BAT = r"D:\tools\colmap-x64-windows-nocuda\COLMAP.bat"   # for open_in_colmap.bat (None: search the usual places)
WORKERS = 4                                  # image-processing worker processes

# --- site ---------------------------------------------------------------------------------------------------
from mppp.sfm.sites import load_sites
SITES_FILE  = None             # v0p43: the site definitions (name -> sol range): None = mppp/data/sites.json
SITES       = load_sites()     # name -> (first sol, last sol), from mppp/data/sites.json (edit the file to add sites)
SITES_EXTRA = {}               # more sites for this run only, e.g. {"my_site": (700, 712)}
SITE = "rockytop"         # difficult case (winter sun, low cross-station match rate)
# SITE = "threeforks"         # easy case, flat ground
# SITE = "butler_landing"
# SITE = "seitah_north"
# SITE = "sid"
# SITE = "whale_mountain"
# SITE = "van_zyl"
# SITE = "olifants"
# SITE = "origny"
# SITE = "groloy"
# SITE = "overlook_mountain"
# SITE = "pico_turquino"
# SITE = "bell_island"
# SITE = "taylorfjellet"
# SITE = "threeforks_south"
# SITE = "rockytop"

# --- where the images come from and where the results go (v0p43) -----------------------------------------------
SOURCE    = "pds"      # "pds": select from PDS_DIR and process (reusing what is already processed);
                       # "processed": align the images already in WORK/processed (its newest manifest) - no PDS search,
                       # no image processing; images deleted from processed/images_png8 are left out
WORK_DIR  = None       # a WORK folder to use instead of SCAPES_ROOT/mars2020_sol_<sol>_<SITE>_colmap[_zcam] (the
                       # site and INCLUDE_ZCAM follow its name; v0p53)
VARIANT   = ""         # "" -> results in WORK/colmap; "name" -> WORK/colmap_name (other SfM settings or camera
                       # models side by side; starts from colmap/'s features and matches)
EXCLUDE   = []         # images to leave out, as in WORK/exclude_images.txt (both apply): "S032D1184" (station),
                       # "sol:658" / "sol:654-693", "seq:NCAM08111", or a file-name pattern like "ZR0_0690_*"
REUSE_MATCHES = True   # v0p43: keep the matches of the previous database.db (same features and matching settings)
RUN_KEY   = None       # set by scripts/align_scape.py and run_sites.py (recorded in <project>/run_done.json)

ADD_NEARBY_WAYPOINTS = 10                    # also take every waypoint station within this many metres (inclusive)
                                             # of a station imaged in the sol range, any sol; 0 = off

# --- processing ---------------------------------------------------------------------------------------------
REPROCESS_ALL          = 0   # True: process every image again (new masks, radiometry, poses); False: reuse
KEEP_ONLY_REMAINING    = False   # v0p53 default False: every selected product is processed and kept (images deleted
                                 # from processed/images_png8 come back), so blocks of several Mastcam-Z zooms reuse the
                                 # same processed files; bad images are left out at the alignment (outlier frames,
                                 # EXCLUDE / exclude_images.txt).  True: only the images still in processed/images_png8
SKY_ELEVATION_DEG      = 20.0    # Navcam frames pointing higher than this (sky surveys) are not processed
LMST_WINDOW_H          = (8.0, 17.0)   # images taken outside this local mean solar time window are not processed
MAX_SATURATED_FRACTION = 0.05    # images with more than this fraction of their valid pixels saturated are not processed
MAX_EXPOSURE_MS        = 40.0    # v0p41: Navcam frames labelled with a longer exposure are not processed: the long
                                 # member of the NCAM08111 exposure brackets (sols 654-693, ~50-65 ms) has a dark blue centre
MAX_CENTRE_TINT        = 1.2     # v0p41: ... nor frames whose centre is bluer than the edge by more than this (B/R ratio)
NAVCAM_BRIGHTNESS      = 0.9     # brightness factor for Navcam (v0p30: 10 % darker to keep nominal exposures in range)
NO_MASK_INFERENCE_AT = [         # stations where the mask model is off (the rover stays in the images)
                            # "S003D0000",    # OEB landing site
                            # "S026D1004",    # Rockytop
                              "S032D1184",    # Three Forks (v0p61 default)
                            # "S047D0000",    # Airey Hill
                        ]
STORE_MASK_IN_ALPHA    = 0.5     # v0p61 default 0.5; also put the mask in the PNG alpha channel (masks/ is written either way): True / 1 =
                                 # masked pixels fully transparent, 0.5 = half transparent (v0p51), False / 0 = no alpha

# --- cameras ------------------------------------------------------------------------------------------------
NAVCAM_DISTORTION = "fisheye_tangential"   # v0p35: "fisheye_tangential" (THIN_PRISM_FISHEYE, sx1 = sy1 = 0: the joint
                                   # calibration's choice, needs NAVCAM_CAMERAS), "rational" (valid to the frame corners;
                                   # NAVCAM_CAMERAS = ...\navcal_v0p40\navcam_joint_rational) or "polynomial" (Metashape K1-K3)
NAVCAM_RIG        = "consensus"    # rig rotation start: "consensus" (refined, v0p22) or "cahv"; baseline = CAHV
NAVCAM_CAMERAS    = None           # folder of verified consensus Navcam cameras from notebook 04 (None: shipped)
from mppp.sfm.project import NAVCAM_CONSENSUS_DIR
NAVCAM_CAMERAS    = NAVCAM_CONSENSUS_DIR   # v0p50: src/mppp/data/cmods, the camera models in use (scripts/promote_cmods.py updates them; MPPP_CMODS overrides); or a folder from notebook 04
NAVCAM_INTRINSICS = "auto"       # "refine" (default: always refine the Navcam intrinsics), "hold" (keep the start cameras), or "auto" (hold where the network is weak: < 4 stations or < 5 m span)
NAVCAM_RIG_REFINE = "refine"         # "refine" (= "rotation": always refine the rig rotation), "auto" (v0p35: refine it
                                   # on a strong network, hold it on a weak one) or "hold"; the baseline is always held.
                                   # A held rig starts at the joint rig turned to the block's camera temperature and sol
NAVCAM_RIG_YAW    = "hold"         # v0p52 default "hold": one constant rig yaw (about y; it shifts the disparity) - the
                                   # rig file's, with no drift or temperature term - held in every block and temperature
                                   # bin; the principal points absorb the block-to-block differences.  "refine" (as
                                   # NAVCAM_RIG_REFINE, per block) or "zero" (set to 0 and held).  Pitch and roll follow
                                   # NAVCAM_RIG_REFINE
NAVCAM_DISTORTION_FIT = "hold"     # v0p50: "hold": the Navcam distortion (k1-k4, p1, p2) stays at NAVCAM_CAMERAS' for every
                                   # block, sol and temperature bin (one set per eye); only f and the principal point are
                                   # refined.  "refine": fitted per block as before v0p50 (TANGENTIAL then decides p1, p2)
NAVCAM_K4         = "zero"         # v0p52 default "zero" (k4 = 0, held; the v0p52 consensus is fitted that way) or
                                   # "consensus" (the start camera's k4).  p1, p2: TANGENTIAL
TANGENTIAL        = "refine"       # Navcam p1, p2 with NAVCAM_DISTORTION_FIT = "refine": "refine", "xml" (hold at the
                                   # calibration) or "zero" (p1 = p2 = 0; with "hold" too)
ZCAM_K3           = "label"        # v0p61: Mastcam-Z k3: "label" (the start camera's, held in the focus bins) or "zero"
                                   # (k3 = 0, as ZCAM_TANGENTIAL = "zero" does for p1, p2)
ZCAM_TANGENTIAL   = "zero"         # v0p40: Mastcam-Z p1, p2: "refine", "xml" (hold at the start camera), "zero"
                                   # (set to 0 and held), or None (as TANGENTIAL); independent of the Navcam setting
INCLUDE_ZCAM      = False          # v0p53: add the Mastcam-Z frames of the same sols (folder ..._colmap_zcam)
ZCAM_ZOOMS        = None           # v0p53: their zooms (mm), e.g. [34, 48]: None = the site's "zcam" list in sites.json
                                   # ([34] for a site without one); "all" = every zoom MPPP aligns (34, 48, 63, 79, 110 mm)
                                   # that the archive / processed/ has for these sols (run_sites_zcam.bat)
INCLUDE_ZCAM34    = None           # before v0p53: True / False sets INCLUDE_ZCAM (and ZCAM_ZOOMS = [34] if None)
ZCAM_INTRINSICS   = "focus_model"  # Mastcam-Z start: "focus_model" (v0p22), "label" or "xml"
                                   # v0p42 focus model: f from focus, the focal-plane temperature (HEAD_FPA; slope 0 for now)
                                   # and sol (clamped to sols 461-991); ZR cx, cy move with focus (+2.5, +1.9 px/1000 counts)
ZCAM_FOCUS_MODEL  = None           # v0p42: a focus-model JSON (e.g. notebook 04 §5b's candidate); None = the one in src/mppp/data/cmods (v0p50; MPPP_CMODS overrides)
ZCAM_FOCUS_BIN    = 30             # focus motor counts per Mastcam-Z camera (None: one camera per eye)
ZCAM_BIN_REFINE   = "focal"        # a focus-bin camera refines fx, fy only ("all": also principal point, distortion;
                                   # v0p53 "model": f held at the zoom's focus model, i.e. linear in focus)
ZCAM_HOLD_F_IMAGES = 2             # focus bins with <= this many images hold f at the focus model
ZCAM_BACKLASH     = "list"         # focus backlash states. v0p61 "list" (default): the images of ZCAM_LOW_BACKLASH are
                                   # the low-backlash (regular) state from the start (own bins, f at the label), all
                                   # others high-backlash (110 mm: one state); after the alignment every focus group is
                                   # still classified from its images' own focal lengths, for comparison
                                   # (zcam_focus_states.json). "split" (v0p40): classify and give the regular images own
                                   # cameras before a last adjustment; "report": classify only; None: off
ZCAM_LOW_BACKLASH = None           # v0p61: the list of low-backlash observations (names or parts of names, sol:sequence);
                                   # None = src/mppp/data/mars2020_mastcam-z_low_backlash_observations.csv; False = no list
ZCAM_FOCUS_LINE   = True           # v0p61: stage 2 holds the Mastcam-Z cameras at the focus model for its first round,
                                   # then ZCAM_LINE_CYCLES times frees the bins' f for a round and puts every bin back on
                                   # one line f0 + slope (focus - ref) per eye and zoom fitted to them (outlier bins
                                   # listed, not followed); the rest holds them on the line. False: f per bin (v0p53)
ZCAM_LINE_CYCLES  = 2
THERMAL_AFTER_STAGE1 = True        # v0p61: the Navcam temperature bins are fitted at the end of stage 1 (Navcam only),
                                   # before the Mastcam-Z frames come in
ZCAM_RIG          = True           # True: the two Mastcam-Z eyes of an exposure form a stereo rig, one per zoom
                                   # shared by every focus bin (v0p53); False: each image is its own frame
ZCAM_RIG_REFINE   = "auto"         # v0p53: "auto" = held at the zoom focus model's rig when it has one, else refined
                                   # (rotation) from the CAHV median; "hold"; "refine"

# --- features and matching ----------------------------------------------------------------------------------
MAX_NUM_FEATURES = 32768         # v0p63 (16384 from v0p50): binds on full-res Navcam frames (19.7 MP), rarely on
                                 # Mastcam-Z (2 MP) - more Navcam features where a Zcam frame overlaps
SIFT = dict(domain_size_pooling=False, estimate_affine_shape=False,   # DSP-SIFT / affine shape (CPU only, slower)
            num_octaves=5, first_octave=-1)  # v0p63: 5 octaves (2x ... 1/8 of the image; COLMAP's default 4 stops at
                                             # 1/4): coarse features on the finer camera for Navcam-Zcam pairs
MATCH_MODE = "exhaustive"          # "exhaustive" (baseline) or "prior_pairs" (quick / CPU)
MATCH = dict(max_ratio=0.9, max_distance=1.0, cross_check=True, guided_matching=True, max_error_px=6.0)   # v0p63: guided
MATCH_BLOCK_SIZE = 100             # images per exhaustive-matching block (COLMAP default 50; larger = fewer reloads)

# --- reconstruction -----------------------------------------------------------------------------------------
# rounds of (triangulation threshold [full-res px], Cauchy scale [sigma], max residual kept [native px])
SCHEDULE = ((24.0, 10.0, 8.0), (12.0, 4.0, 4.0), (8.0, 2.0, 2.0))
MIN_TRI_ANGLE_DEG = 0.25           # keeps stereo-only points out to ~97 m at the 0.42 m baseline
TRIANGULATION = dict(max_transitivity=1)     # COLMAP track building (see mppp.sfm.reconstruction.triangulate)
ATTITUDE_PRIOR_DEG = 2.0           # CAHV attitude prior per frame (1 deg holds the ENU frame more tightly)
LOCALIZE_MIN_IMAGES = 3            # stations (site, drive) with fewer images get no waypoint position prior
EXCLUDE_OUTLIERS = True            # drop frames that do not align with the majority, then re-adjust
OUTLIER_THRESHOLDS = {}            # e.g. {"residual_factor": 3.0, "min_shift_m": 0.25} (reconstruction.OUTLIER_DEFAULTS)
MIN_FRAME_OBSERVATIONS = 20        # v0p43.3: frames with fewer tie-point observations are held at their prior and
                                   # excluded as outliers (was 30; e.g. 12 keeps more weakly tied frames)

STAGED            = True           # Navcam block first, then Mastcam-Z with the Navcam cameras and rig held
HOLD_CAMERAS      = ()             # further camera keys to hold at their start values, e.g. ("ZL034_f0",)
LINEAR_SOLVER     = "auto"         # bundle adjustment: "auto" (dense Schur up to 600 frames, else iterative), or
                                   # "sparse_schur" / "dense_schur" / "iterative_schur"
THERMAL_BINS_DEG  = 5              # v0p44 (was 10): Navcam camera-temperature bins for the final stage - one camera per eye
                                   # and bin; None: off.  Bins are fixed multiples of this (..., -20..-15, -15..-10 degC);
                                   # a frame goes by the mean temperature of its Navcam images
THERMAL_MIN_IMAGES = 5             # v0p44 (was 8): a bin with fewer images (both eyes counted) is merged into the
                                   # neighbouring occupied bin whose centre is nearer (smallest first, repeated)
THERMAL_FREE = ("fx", "fy")        # v0p44: refined per bin (unless the Navcam intrinsics are held); cx, cy stay at the
                                   # eye's refined values moved by the thermal model (NL cx +0.052 px/degC). Adding
                                   # "cx", "cy" fits them per bin too (the rig is held in this stage; weak in small bins)
                                   # v0p50: only fx, fy, cx, cy - the distortion is never fitted per temperature bin

In [ ]:
# derived settings
from mppp.sfm.sites import check_sites, load_sites, parse_work_folder
from mppp.sfm.workdir import project_dir, load_manifest, sol_range
SITES = {**(load_sites(SITES_FILE) if SITES_FILE else SITES), **{k: tuple(v) for k, v in SITES_EXTRA.items()}}
if WORK_DIR:                                      # v0p43: an existing WORK folder names the site
    WORK_DIR = Path(WORK_DIR)
    _site, _z = parse_work_folder(WORK_DIR)
    SITE = _site or WORK_DIR.name
    if _site:
        INCLUDE_ZCAM = _z
if INCLUDE_ZCAM34 is not None:                    # before v0p53
    INCLUDE_ZCAM = bool(INCLUDE_ZCAM34)
    if INCLUDE_ZCAM34 and ZCAM_ZOOMS is None:
        ZCAM_ZOOMS = [34]
if INCLUDE_ZCAM and ZCAM_ZOOMS is None:            # v0p53: the site's Mastcam-Z zooms
    from mppp.sfm.sites import load_site_table, site_zooms
    ZCAM_ZOOMS = site_zooms(load_site_table(SITES_FILE)["sites"].get(SITE)) or [34]
if isinstance(ZCAM_ZOOMS, str) and ZCAM_ZOOMS.lower() == "all":    # v0p53: whatever is on disk
    ZCAM_ZOOMS = [34, 48, 63, 79, 110]
ZCAM_ZOOMS = sorted(int(z) for z in (ZCAM_ZOOMS or [])) if INCLUDE_ZCAM else []
if set(ZCAM_ZOOMS) - {34, 48, 63, 79, 110}:
    raise ValueError(f"ZCAM_ZOOMS = {ZCAM_ZOOMS}: MPPP aligns the Mastcam-Z zooms 34, 48, 63, 79 and 110 mm")
if SITE not in SITES:
    if SOURCE == "processed" and WORK_DIR:        # a folder of any name: the sol range of its processed images
        SITES[SITE] = sol_range(load_manifest(Path(WORK_DIR) / "processed")[0]["images"])
    else:
        raise KeyError(f"site {SITE!r} is not in {SITES_FILE or 'mppp/data/sites.json'} or SITES_EXTRA")
if NAVCAM_CAMERAS and not (Path(NAVCAM_CAMERAS) / "M2020_N_rig.json").is_file():   # v0p43.2
    raise FileNotFoundError(f"NAVCAM_CAMERAS = {NAVCAM_CAMERAS}: no consensus Navcam cameras there (moved?). The default "
                            f"is src/mppp/data/cmods (mppp.sfm.project.NAVCAM_CONSENSUS_DIR)")
for _k, _v, _ok in (("NAVCAM_DISTORTION_FIT", NAVCAM_DISTORTION_FIT, ("hold", "refine")),     # v0p50
                    ("NAVCAM_K4", NAVCAM_K4, ("consensus", "zero")),
                    ("NAVCAM_RIG_YAW", NAVCAM_RIG_YAW, ("hold", "refine", "zero"))):
    if _v not in _ok:
        raise ValueError(f"{_k} = {_v!r}: use one of {_ok}")
if SOURCE not in ("pds", "processed"):
    raise ValueError(f"SOURCE = {SOURCE!r}: use 'pds' or 'processed'")
for _w in check_sites(SITES):
    print("WARNING: SITES", _w)
SOLS = SITES[SITE]
if SOLS[1] < SOLS[0]:
    raise ValueError(f"SITES[{SITE!r}] = {SOLS}: the end sol is before the start sol")
RIG_REFINE = {"refine": "rotation", "rotation": "rotation", "auto": "auto", "hold": False}.get(NAVCAM_RIG_REFINE)
if RIG_REFINE is None:
    raise ValueError(f"NAVCAM_RIG_REFINE = {NAVCAM_RIG_REFINE!r}: use 'refine', 'auto' or 'hold'")
from mppp.sfm.sites import work_folder
WORK = Path(WORK_DIR) if WORK_DIR else work_folder(SCAPES_ROOT, SITE, bool(INCLUDE_ZCAM), SOLS)   # v0p53: mars2020_sol_...
for _k, _v in (("TANGENTIAL", TANGENTIAL), ("ZCAM_TANGENTIAL", ZCAM_TANGENTIAL)):
    if _v not in ("refine", "xml", "zero") and not (_k == "ZCAM_TANGENTIAL" and _v is None):
        raise ValueError(f"{_k} = {_v!r}: use 'refine', 'xml' or 'zero'")
_ZT = ZCAM_TANGENTIAL or TANGENTIAL
ZERO_TERMS = ("p1", "p2", "b1", "b2") if TANGENTIAL == "zero" else ("b1", "b2")              # Navcam
ZCAM_ZERO_TERMS = None if _ZT == TANGENTIAL else (("p1", "p2", "b1", "b2") if _ZT == "zero" else ("b1", "b2"))
if ZCAM_K3 not in ("label", "zero"):
    raise ValueError(f"ZCAM_K3 = {ZCAM_K3!r}: use 'label' or 'zero'")
if ZCAM_K3 == "zero":                              # v0p61
    ZCAM_ZERO_TERMS = tuple(ZCAM_ZERO_TERMS if ZCAM_ZERO_TERMS is not None else ZERO_TERMS) + ("k3",)
# p1, p2 refined per camera family (v0p40): a bool when both follow TANGENTIAL
REFINE_TANGENTIAL = ((TANGENTIAL == "refine") if _ZT == TANGENTIAL
                     else {"N": TANGENTIAL == "refine", "Z": _ZT == "refine"})
if GPU_PY:
    print("GPU env:", check_gpu_python(GPU_PY))
print(SITE, SOLS, "->", WORK, "| Mastcam-Z:", ZCAM_ZOOMS or "no", "| reprocess all:", bool(REPROCESS_ALL),
      "| source:", SOURCE, "| variant:", VARIANT or "(default)")
COLMAP_DIR = project_dir(WORK, VARIANT)            # WORK/colmap, or WORK/colmap_<VARIANT>
# v0p35.1: a WORK folder left by another sol range (e.g. a site renamed in SITES) would mix two blocks
if (COLMAP_DIR / "project.json").is_file():
    _sols = [int(r["sol"]) for r in json.loads((COLMAP_DIR / "project.json").read_text()).get("images", []) if "sol" in r]
    _in = sum(SOLS[0] <= s <= SOLS[1] for s in _sols)
    if _sols and _in == 0:
        print(f"WARNING: {COLMAP_DIR} holds a block of sols {min(_sols)}-{max(_sols)}, none in {SOLS[0]}-{SOLS[1]}: "
              f"rename or move that folder before running {SITE!r} with this sol range")


def banner():                          # first line of every results cell: the site and its COLMAP folder
    print(f"== {SITE}  (sols {SOLS[0]}-{SOLS[1]})  |  {COLMAP_DIR}")

## 1–2. Select and process

In [ ]:
banner()
if SOURCE == "pds":
    if ADD_NEARBY_WAYPOINTS:              # the sol range plus other visits within ADD_NEARBY_WAYPOINTS metres of its stations
        from mppp.select import find_imgs_near
        WAYPOINTS = mppp.load_waypoints()
        paths, near = find_imgs_near(PDS_DIR, ["NLF", "NRF"], SOLS, WAYPOINTS, radius_m=ADD_NEARBY_WAYPOINTS)
        print(f"Navcam: {near['n_in_range']} in sols {SOLS[0]}-{SOLS[1]} + {near['n_added']} from other visits within "
              f"{ADD_NEARBY_WAYPOINTS} m")
        for a in near["stations_added"]:
            print(f"   {a['station']} ({a['distance_m']} m from {a['nearest']}): {a['images']} images, sols {a['sols'][0]}-{a['sols'][1]}")
    else:
        paths = find_imgs(PDS_DIR, ["NLF", "NRF"], sol_range=SOLS)
    prefixes = ("NCAM",)
    for _zoom in ZCAM_ZOOMS:                      # v0p53: every zoom of the site (sequ_id "_034", "_048", "_063", "_110")
        if ADD_NEARBY_WAYPOINTS:
            # v0p53.1: from the Navcam block's stations, so every zoom gets the same visits (e.g. Sid sols 91-101)
            zp, znear = find_imgs_near(PDS_DIR, ["ZL0", "ZR0"], SOLS, WAYPOINTS, radius_m=ADD_NEARBY_WAYPOINTS,
                                       sequ_id=f"_{_zoom:03d}", anchor_stations=near["stations_in_range"])
            print(f"Mastcam-Z {_zoom}: {znear['n_in_range']} in range + {znear['n_added']} from other visits")
            paths += zp
        else:
            zp = find_imgs(PDS_DIR, ["ZL0", "ZR0"], sol_range=SOLS, sequ_id=f"_{_zoom:03d}")
            print(f"Mastcam-Z {_zoom}: {len(zp)} products")
            paths += zp
    if ZCAM_ZOOMS:
        prefixes += ("ZCAM",)
    paths, sel = select_best_products(paths, sequence_prefix=prefixes)      # one product per exposure
    print(json.dumps({k: v for k, v in sel.items() if k != "dropped_subframes"}, indent=1)); print(len(paths), "products")
    if sel.get("n_dropped_subframes"):             # v0p43.3: single full-resolution Navcam tiles (the full frame is kept)
        print(sel["n_dropped_subframes"], "Navcam tiles / sub-frames below", sel["min_frame_fraction"],
              "of the frame left out, e.g.", sel["dropped_subframes"][0]["file"])
else:
    paths, sel = None, None
    print("SOURCE = 'processed': no PDS selection; the images already in", WORK / "processed", "are aligned")


In [ ]:
banner()
cfg = mppp.default_config()
cfg["export"].update(formats=["PNG8"], store_mask_in_alpha=STORE_MASK_IN_ALPHA, write_mask_files=True,
                     write_colmap=False)
# mask model: the released default ("mppp_mask_v2"); or cfg["masking"]["checkpoint"] = "<path to your checkpoint>"
cfg["masking"]["skip_inference_at"] = NO_MASK_INFERENCE_AT
cfg["selection"].update(max_boresight_elevation_deg=SKY_ELEVATION_DEG,       # sky-pointing Navcam frames
                        lmst_window_h=list(LMST_WINDOW_H) if LMST_WINDOW_H else None,
                        max_saturated_fraction=MAX_SATURATED_FRACTION,
                        max_exposure_ms=MAX_EXPOSURE_MS, max_centre_tint=MAX_CENTRE_TINT)
cfg["color"]["brightness_by_family"] = {"N": NAVCAM_BRIGHTNESS}
PROCESSED = WORK / "processed"
# KEEP_ONLY_REMAINING = False: every selected product (images deleted from processed/images_png8 come back).
# REPROCESS_ALL = False: images processed with the same configuration are reused from the manifest.
if SOURCE == "pds":
    manifest = mppp.process_images(paths, PROCESSED, cfg, mppp.load_waypoints(), provenance=sel,
                                   only_existing="PNG8" if KEEP_ONLY_REMAINING else None,
                                   reuse_existing=not REPROCESS_ALL, workers=WORKERS)
    print(manifest["n_processed"], "images in the manifest,", len(manifest["failed"]), "failed,",
          len(manifest.get("skipped", [])), "skipped by the selection rules (sky, LMST window, saturation, exposure)")
    skipped = [m["source_product"] for m in manifest["images"] if m["mask"].get("inference_skipped")]
    if skipped:
        print(len(skipped), "images without mask inference (rover visible):",
              sorted({f"S{m['site']:03d}D{m['drive']:04d}" for m in manifest["images"] if m["mask"].get("inference_skipped")}))
    if manifest.get("only_existing"):
        print("kept", manifest["only_existing"]["n_kept"], "of", manifest["only_existing"]["n_selected"], "selected;",
              manifest["only_existing"]["n_removed"], "removed")
else:                                              # v0p43: the images already processed in this WORK folder
    from mppp.sfm.workdir import keep_existing
    manifest, _mf = load_manifest(PROCESSED)
    manifest = dict(manifest)
    _n0 = len(manifest["images"])
    if KEEP_ONLY_REMAINING:
        manifest["images"], _gone = keep_existing(manifest["images"], PROCESSED, "PNG8")
        if _gone:
            print(len(_gone), "images of the manifest are no longer in processed/images_png8 - left out")
    print(f"{len(manifest['images'])} of {_n0} images from {_mf.name}")
# v0p43: images left out by WORK/exclude_images.txt and EXCLUDE (nothing is deleted)
from mppp.sfm.workdir import EXCLUDE_FILE, read_exclusions, apply_exclusions
_entries = read_exclusions(WORK / EXCLUDE_FILE) + list(EXCLUDE)
if _entries:
    manifest = dict(manifest)
    manifest["images"], _removed = apply_exclusions(manifest["images"], _entries)
    _by = {}
    for _r in _removed:
        _by.setdefault(_r["entry"], []).append(_r["image"])
    for _e, _ims in _by.items():
        _ims = [x for x in _ims if x]
        print(f"excluded by {_e!r}: {len(_ims)} images" + ("" if _ims else "  <- matches no image (typo?)"))
    print(len(manifest["images"]), "images after the exclusions")


## 3–4. Project, features, database

In [ ]:
from mppp.sfm.backlash import low_backlash_fingerprint                                   # v0p61
banner()
from collections import Counter
import numpy as np
PROJECT = COLMAP_DIR                               # WORK/colmap or WORK/colmap_<VARIANT>
REUSE_FROM = []
if VARIANT:                                        # v0p43: a variant starts from colmap/'s features and matches
    from mppp.sfm.workdir import seed_variant
    _seed = seed_variant(PROJECT, project_dir(WORK, None))
    REUSE_FROM = _seed["reuse_matches_from"] if REUSE_MATCHES else []
    if _seed["copied"]:
        print("variant", VARIANT, ": features copied from", _seed["base"])
from mppp.sfm.project import navcam_cameras_fingerprint
proj = SfmProject.load(PROJECT) if (PROJECT / "project.json").is_file() else None
wanted = {"zero_terms": list(ZERO_TERMS), "zcam_intrinsics": ZCAM_INTRINSICS,
          "zcam_zero_terms": list(ZCAM_ZERO_TERMS) if (ZCAM_ZERO_TERMS is not None and INCLUDE_ZCAM) else None,
          "zcam_focus_bin": float(ZCAM_FOCUS_BIN) if ZCAM_FOCUS_BIN else None, "zcam_bin_refine": ZCAM_BIN_REFINE,
          "zcam_rig": ZCAM_RIG, "zcam_rig_refine": ZCAM_RIG_REFINE,                                   # v0p53
          "zcam_low_backlash_sha256": low_backlash_fingerprint(ZCAM_LOW_BACKLASH) if INCLUDE_ZCAM else None,  # v0p61
          "navcam_distortion": NAVCAM_DISTORTION, "navcam_rig": NAVCAM_RIG,
          "zcam_hold_f_images": int(ZCAM_HOLD_F_IMAGES) if ZCAM_INTRINSICS == "focus_model" else None,
          "prior_R_corrected": True,
          "navcam_distortion_fit": NAVCAM_DISTORTION_FIT, "navcam_k4": NAVCAM_K4,                           # v0p50
          "navcam_rig_yaw": NAVCAM_RIG_YAW}                                                                  # v0p51
# v0p53: only the Mastcam-Z zooms of this run (ZCAM_ZOOMS; none without INCLUDE_ZCAM)
_nz = len(manifest["images"])
manifest["images"] = [m for m in manifest["images"] if "failed" in m or m.get("filename", {}).get("family") != "Z"
                      or m["filename"].get("zoom_mm") in ZCAM_ZOOMS]
if len(manifest["images"]) < _nz:
    print(f"{_nz - len(manifest['images'])} Mastcam-Z images of other zooms than {ZCAM_ZOOMS} left out")
names = {Path(m["source_product"]).stem for m in manifest["images"]}
if (REPROCESS_ALL or proj is None or names != {r["stem"] for r in proj.images}    # other images than the manifest
        or any(proj.settings.get(k) != v for k, v in wanted.items())            # or different camera settings
        or (proj.settings.get("navcam_cameras") or {}).get("dir") != (str(NAVCAM_CAMERAS) if NAVCAM_CAMERAS else None)
        or (NAVCAM_CAMERAS and (proj.settings.get("navcam_cameras") or {}).get("fingerprint")
            != navcam_cameras_fingerprint(NAVCAM_CAMERAS))):     # v0p31: a rewritten consensus rebuilds the project
    proj = SfmProject.create(manifest["images"], PROCESSED, PROJECT, prior_sigma_m=(1.0, 1.0, 1.0),
                             zcam_intrinsics=ZCAM_INTRINSICS, zero_terms=ZERO_TERMS,
                             zcam_focus_bin=ZCAM_FOCUS_BIN, zcam_bin_refine=ZCAM_BIN_REFINE, zcam_rig=ZCAM_RIG,
                             navcam_distortion=NAVCAM_DISTORTION, navcam_rig=NAVCAM_RIG,
                             zcam_hold_f_images=ZCAM_HOLD_F_IMAGES,
                             navcam_cameras=NAVCAM_CAMERAS,
                             zcam_zero_terms=ZCAM_ZERO_TERMS if INCLUDE_ZCAM else None, zcam_rig_refine=ZCAM_RIG_REFINE,
                             zcam_low_backlash=ZCAM_LOW_BACKLASH,                                       # v0p61
                             zcam_focus_model_file=ZCAM_FOCUS_MODEL,
                             navcam_distortion_fit=NAVCAM_DISTORTION_FIT, navcam_k4=NAVCAM_K4,
                             navcam_rig_yaw=NAVCAM_RIG_YAW)
else:
    from mppp.sfm.thermal import strip_thermal_bins
    strip_thermal_bins(proj)                     # v0p43: back to one camera per eye / focus bin before a rerun
    proj.refresh_images(manifest["images"])      # copies (no hard links) follow images processed again
    # v0p42: a project built with another Mastcam-Z focus model keeps its bins (an alignment is expensive); say so
    if INCLUDE_ZCAM and ZCAM_INTRINSICS == "focus_model":
        from mppp.sfm.project import zcam_focus_model_fingerprint
        _fm = (proj.settings.get("zcam_focus_model") or {}).get("fingerprint")
        if _fm != zcam_focus_model_fingerprint(ZCAM_FOCUS_MODEL):
            print("NOTE: this project's Mastcam-Z focus bins were built with "
                  + ("an earlier focus model" if _fm is None else "another focus model")
                  + "; set REPROCESS_ALL = True to rebuild them from the current one (a new alignment)")
if COLMAP_BAT:
    proj.settings["colmap_bat"] = COLMAP_BAT     # open_in_colmap.bat starts this COLMAP
    proj.save()
print(len(proj.images), "images;", {k: round(v["baseline_m"], 4) for k, v in proj.rig.items()}, "m baseline;",
      {k: v["n_pairs"] for k, v in proj.rig.items()}, "CAHV pairs;",
      {k: v.get("rotation_source", "CAHV") for k, v in proj.rig.items()}, "rig rotation")
for k, c in proj.cameras.items():
    if not c.get("group"):
        print(k, [round(x, 3) for x in c["params"][:5]], f"p1 {c['params'][6]:.2e} p2 {c['params'][7]:.2e}",
              ("distortion held" if c.get("distortion_fit") == "hold" else "") +
              (f" ({', '.join(c['zeroed_terms'])} = 0)" if c.get("zeroed_terms") else ""),
              f"k3 {c['params'][8]:.3g}", f"k4 {c['params'][9]:.3g} (rational)" if c.get("free_params") else "")
bins = Counter(c["group"] for c in proj.cameras.values() if c.get("group") and "focus_count_range" in c)
for g, n in sorted(bins.items()):
    cs = [c for c in proj.cameras.values() if c.get("group") == g and "focus_count_range" in c]
    held = sum("fx" in (c.get("fixed_params") or []) for c in cs)
    print(f"{g}: {n} focus bins ({held} with f held), counts {min(c['focus_count_range'][0] for c in cs):.0f}-"
          f"{max(c['focus_count_range'][1] for c in cs):.0f}, images per bin {sorted(c['n_images'] for c in cs)}")
corr = [r["prior_R_correction_deg"] for r in proj.images if "prior_R_correction_deg" in r]
if corr:   # Mastcam-Z label attitudes rotated to fit the camera's principal point (v0p14.5)
    print(f"Mastcam-Z prior attitude correction: median {np.median(corr):.3f} deg, max {max(corr):.3f} deg ({len(corr)} images)")
print("images per camera:", dict(Counter(r["instrument"] for r in proj.images)))
# v0p43: every setting of this run, beside the results
_settings = {k: repr(v) for k, v in sorted(globals().items())
             if k.isupper() and not k.startswith("_") and isinstance(v, (str, int, float, bool, tuple, list, dict, type(None), Path))}
(proj.root / "run_settings.json").write_text(json.dumps({"mppp": mppp.__version__, "run_key": RUN_KEY, "settings": _settings},
                                                        indent=1), encoding="utf-8")


In [ ]:
banner()
# SIFT at native resolution with masks -> features.db.  Reused on reruns only if features.json beside it records
# the same settings and image/mask files; overwrite=True always redoes it.
extract_features(proj, max_num_features=MAX_NUM_FEATURES, use_gpu=True if GPU_PY else None, python=GPU_PY,
                 overwrite=bool(REPROCESS_ALL), **SIFT)
# replaces database.db on every run; v0p43: its matches (and a variant's base matches) are carried over when the
# features and matching settings are the same - COLMAP then matches only the pairs that are missing
from mppp.sfm.matching import match_settings
_db = build_database(proj, reuse_matches_from=REUSE_FROM,
                     matching=match_settings(MATCH_MODE, **MATCH) if REUSE_MATCHES else None)
if _db.get("reused_matches"):
    print("matches reused:", {k: _db["reused_matches"][k] for k in ("matches", "geometries")},
          [s.get("skipped") or "used" for s in _db["reused_matches"]["sources"]])
print({k: v for k, v in _db.items() if k != "reused_matches"})

## 5. Matching

In [ ]:
banner()
if MATCH_MODE == "prior_pairs":
    pairs = prior_overlap_pairs(proj, min_overlap=0.05)
    print(len(pairs), "prior pairs")
print(match(proj, mode=MATCH_MODE, use_gpu=True if GPU_PY else None, python=GPU_PY, block_size=MATCH_BLOCK_SIZE, **MATCH))

## 6. Reconstruct and refine

In [ ]:
banner()
from mppp.sfm.reconstruction import navcam_network
net = navcam_network(proj)
print(f"Navcam network: {net['stations']} stations over {net['span_m']:.1f} m, {net['navcam_images']} images -> {net['verdict']}"
      f" (intrinsics setting {NAVCAM_INTRINSICS!r})")
from mppp.sfm.thermal import image_temperatures, thermal_model_for_project
TEMPS = image_temperatures(proj, pds_dir=PDS_DIR if THERMAL_BINS_DEG else None, verbose=bool(THERMAL_BINS_DEG))
rec = reconstruct(proj, sigma_px=0.5,           # keypoint sigma in native pixels
                  schedule=SCHEDULE, refine_rig=RIG_REFINE,
                  refine_tangential=REFINE_TANGENTIAL,   # Navcam TANGENTIAL, Mastcam-Z ZCAM_TANGENTIAL
                  min_track_length=2,           # two-view tie points (e.g. seen only by one stereo pair) are kept
                  min_tri_angle_deg=MIN_TRI_ANGLE_DEG, attitude_prior_deg=ATTITUDE_PRIOR_DEG,
                  triangulation_options=TRIANGULATION,
                  exclude_outliers=EXCLUDE_OUTLIERS, outlier_thresholds=OUTLIER_THRESHOLDS,
                  min_frame_observations=MIN_FRAME_OBSERVATIONS,
                  navcam_intrinsics=NAVCAM_INTRINSICS, staged=STAGED, hold_cameras=HOLD_CAMERAS,
                  linear_solver=LINEAR_SOLVER, temperatures=TEMPS, thermal_bins_deg=THERMAL_BINS_DEG,
                  thermal_min_images=THERMAL_MIN_IMAGES, thermal_free=THERMAL_FREE, thermal_model=thermal_model_for_project(proj),
                  localize_min_images=LOCALIZE_MIN_IMAGES,
                  zcam_backlash=ZCAM_BACKLASH if INCLUDE_ZCAM else None,
                  zcam_focus_line=bool(ZCAM_FOCUS_LINE), zcam_line_cycles=int(ZCAM_LINE_CYCLES),     # v0p61
                  thermal_after_stage1=bool(THERMAL_AFTER_STAGE1))
rs = proj.settings["reconstruction"]
print("Navcam intrinsics:", rs.get("navcam_intrinsics"), "| held cameras:", rs.get("hold_cameras") or "none",
      "| staged:", rs.get("staged"))
excluded = rs.get("excluded", [])
print(len(excluded), "frames excluded as outliers")
for st, v in (rs.get("unlocalized") or {}).items():
    print(f"   no position prior: {st} ({v['images']} images, {v['cross_points']} cross-station points): {v['prior']}")
for e in excluded:
    print("  ", ", ".join(e["images"]), "|", e["station"], "|", "; ".join(e["reasons"]))
summary = export_for_error(proj, rec)
print(json.dumps({k: summary[k] for k in ("tracks", "residual_rms_native_px", "registered_images", "project_images",
                                           "prior_offset_median_m", "station_components")}, indent=1))
summary_nav = None
if rs.get("navcam_stage"):
    # the stage-1 Navcam-only solution: same images, features and database as the combined one
    import pycolmap
    rec_nav = pycolmap.Reconstruction(str(proj.root / rs["navcam_stage"]["path"]))
    summary_nav = export_for_error(proj, rec_nav, out_dir=proj.root / "error_input_navcam")
    print("Navcam-only stage exported to error_input_navcam/:",
          {k: summary_nav[k] for k in ("tracks", "residual_rms_native_px", "registered_images")})
conv = convergence_statistics(rec, proj)
proj.settings["reconstruction"]["convergence"] = conv
proj.save()
print("tie points by convergence angle", conv["bins_deg"], "deg:", conv["points"],
      "| cross-station:", conv["cross_station_points"], f"| > 10 deg: {conv['points_over_10deg']}, "
      f"> 20 deg: {conv['points_over_20deg']}")
# start -> refined camera parameters (Mastcam-Z focus bins: section 7b).  Cameras are looked up by database id:
# after the thermal stage an eye's camera can have no images left (all frames in temperature bins).
from mppp.sfm.reconstruction import print_camera_changes
print_camera_changes(rec, proj)
print("COLMAP GUI: double-click", proj.root / "open_in_colmap.bat")
_bl = proj.settings.get("zcam_backlash")
if _bl:   # v0p40: Mastcam-Z focus states (per image: colmap/zcam_focus_states.json)
    print("Mastcam-Z focus groups:", _bl["group_counts"], "| f / label f:",
          {s: (round(v["ratio_median"], 4) if v["ratio_median"] else None) for s, v in (_bl.get("after") or _bl["before"]).items()},
          "| regular-state cameras:", [(r["camera"], r["images"], round(r.get("f_refined_px", r["start_f_px"]), 1)) for r in _bl["cameras"]])


## 7. Alignment health — check before any analysis
Each check has provisional warn and fail thresholds. These were set a priori and are not validated limits; pass `thresholds={...}` to change them. The report is written to `<project>/health/` (`health.json`, `health.md`, `health.png`, and `station_map.png`: a top-down view of the stations with the prior-to-refined shift of every camera).

| section | checks |
|---|---|
| registration | images registered; frames held at their prior (< 30 observations) |
| tie points | points; track-length population (two-view share); observations per image; cross-station ties; tied station blocks |
| reprojection | median and 95th-percentile residual (native px); left vs right eye; outlier images; residual growth towards the image edge; tie-point coverage by image radius |
| camera model | change of f, principal point and k1–k4; image displacement of the same ray |
| stereo rig | right-camera rotation vs its start; spread of the CAHV pairs |
| poses | refined − prior centre per station; spread within a station; attitude change; scale vs the priors |

In [ ]:
banner()
if "rec" not in globals():         # v0p40: after a kernel restart, read the alignment section 6 saved
    from mppp.sfm.reconstruction import load_alignment
    rec = load_alignment(proj)       # a clear error if section 6 has not finished for this project
report = assess_alignment(proj, rec)
print(health_table(report))
files = write_health(report, proj.root / "health", rec, proj)
from IPython.display import Image, display
display(Image(filename=files["png"]))
if files.get("station_map"):                       # top-down view: stations, prior -> refined shifts (health/station_map.png)
    display(Image(filename=files["station_map"]))
if report["verdict"] == "fail":
    print("\nFAIL: fix the alignment (matching, priors, masks, thresholds) before any error analysis.")

### 7b. Mastcam-Z focus breathing
Written to `<project>/health/zcam_focus_breathing.{png,csv,json}`. Notebook 04 compares scapes.

In [ ]:
banner()
if "rec" not in globals():         # v0p40: after a kernel restart, read the alignment section 6 saved
    from mppp.sfm.reconstruction import load_alignment
    rec = load_alignment(proj)       # a clear error if section 6 has not finished for this project
from mppp.sfm.zcam import write_focus_breathing
if any(c.get("group") for c in proj.cameras.values()):
    fb = write_focus_breathing(proj, rec)
    for g, f in fb["fits"].items():
        for kind in ("refined", "label"):
            if f[kind]:
                print(f"{g} {kind:7s}: f = {f[kind]['f0_px']:.1f} px at count {f['reference_count']:.0f}, "
                      f"slope {f[kind]['slope_px_per_count']:+.4f} px/count, rms {f[kind]['rms_px']:.2f} px, n = {f[kind]['n']}")
    display(Image(filename=fb["png"]))
else:
    print("no Mastcam-Z focus-bin cameras")

## 8. Export for the error analysis
Notebook 04 (`04_camera_models.ipynb`) compares the camera models of the sites and writes the consensus cameras; notebook 05 (`05_error_analysis.ipynb`) then reads the `error_input/` folders.

In [ ]:
banner()
import csv
import matplotlib.pyplot as plt
from mppp.sfm.export import plot_camera_shifts
pose_rows = list(csv.DictReader(open(proj.root / "error_input" / "poses.csv")))
fig = plot_camera_shifts(proj, rows=pose_rows, out_png=proj.root / "error_input" / "camera_shifts.png")
plt.close(fig)                                     # the same figure is in health/station_map.png (section 7)
if report["verdict"] == "fail":
    print("alignment health is FAIL (section 7): fix the alignment before using error_input/ in notebook 05")
eps = summary.get("eps", {})
print("error_input:", proj.root / "error_input")
print(f"eps (rms residual / sqrt 2): same-station tracks {eps.get('eps_intra_px', float('nan')):.3f} px, "
      f"cross-station tracks {eps.get('eps_cross_px', float('nan')):.3f} px")
# v0p43: the run finished (scripts/run_sites.py skips a site whose run_done.json has the same RUN_KEY)
import datetime
(proj.root / "run_done.json").write_text(json.dumps({
    "mppp": mppp.__version__, "run_key": RUN_KEY, "site": SITE, "variant": VARIANT, "source": SOURCE,
    "finished": datetime.datetime.now().isoformat(timespec="seconds"), "verdict": report.get("verdict"),
    "images": len(proj.images), "registered_images": summary.get("registered_images"),
    "residual_rms_native_px": summary.get("residual_rms_native_px")}, indent=1, default=str), encoding="utf-8")
print("done:", proj.root / "run_done.json")
